# Unified Sentiment & Emotion Classification Inference Pipeline
This notebook binds all the trained model components together to form a single end-to-end multimodal classification system as specified in the project's proposed architecture.

Given an input text review, audio speech sample, and/or facial image, the pipeline performs:
1. **Preprocessing** (Cleaning, normalization, tokenization, image transforms).
2. **Feature Extraction** (DistilBERT embeddings for text, acoustic stats for audio, CNN feature maps for images).
3. **Model Prediction** (using trained models loaded from the `weights` directory).
4. **Sentiment & Emotion Output** as depicted in the project architecture:
   - **Sentiment**: Positive, Neutral, or Negative.
   - **Emotion**: Happiness, Sadness, Anger, Fear, Surprise, Disgust, or Neutral.

## Setup & Imports

In [ ]:
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import librosa
import joblib
import torch
import torch.nn as nn
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel

# Append current directory to path to import local modules
# Append robust search paths for local modules to support different launch directories
for p in ['.', '..', 'ml-training', '../ml-training']:
    path_to_add = os.path.abspath(p)
    if path_to_add not in sys.path:
        sys.path.append(path_to_add)
from models import TextModel, ImageCNN, AudioLSTM

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Using device: {device}")

## 1. Load Trained Models & Weights

In [ ]:
weights_dir = '../weights' if os.path.exists('../weights') else 'weights'

# 1. Load Text Sentiment Model
text_sentiment_model = TextModel(pretrained_dim=768, feature_dim=256).to(device)
text_sentiment_path = os.path.join(weights_dir, 'text_model.pt')
if os.path.exists(text_sentiment_path):
    text_sentiment_model.load_state_dict(torch.load(text_sentiment_path, map_location=device))
    print(f"Loaded Text Sentiment Model from {text_sentiment_path}")
text_sentiment_model.eval()

# 2. Load Text Emotion Model
text_emotion_model = TextModel(pretrained_dim=768, feature_dim=256).to(device)
text_emotion_path = os.path.join(weights_dir, 'text_model_emotion.pt')
if os.path.exists(text_emotion_path):
    text_emotion_model.load_state_dict(torch.load(text_emotion_path, map_location=device))
    print(f"Loaded Text Emotion Model from {text_emotion_path}")
text_emotion_model.eval()

# 3. Load Image Model
image_model = ImageCNN(feature_dim=256).to(device)
image_model_path = os.path.join(weights_dir, 'image_model.pt')
if os.path.exists(image_model_path):
    image_model.load_state_dict(torch.load(image_model_path, map_location=device))
    print(f"Loaded Image Model from {image_model_path}")
image_model.eval()

# 4. Load Audio Ensemble Classifier & Scaler
audio_model_path = os.path.join(weights_dir, 'audio_ensemble.joblib')
audio_scaler_path = os.path.join(weights_dir, 'audio_scaler.joblib')
if os.path.exists(audio_model_path) and os.path.exists(audio_scaler_path):
    audio_ensemble = joblib.load(audio_model_path)
    audio_scaler = joblib.load(audio_scaler_path)
    print(f"Loaded Audio Ensemble classifier and scaler successfully from weights/")

# Load DistilBERT tokenizer and backbone for text feature extraction
print("Loading DistilBERT tokenizer and backbone encoder...")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
bert_backbone = AutoModel.from_pretrained('distilbert-base-uncased').to(device)
bert_backbone.eval()

## 2. Define Classifier Heads & Mappings
Since the deep feature extractors (`TextModel`, `ImageCNN`) output 256-dimensional feature representations, we define classifier heads to project them to class outputs.

In [ ]:
# Label lists
sentiment_classes = ['Negative', 'Neutral', 'Positive']
emotion_classes = ['Happiness', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Disgust', 'Neutral']

# Initialize demo classification projections
text_sent_classifier = nn.Linear(256, 3).to(device)
text_emot_classifier = nn.Linear(256, 7).to(device)
image_classifier = nn.Linear(256, 3).to(device)

torch.manual_seed(42)
nn.init.xavier_uniform_(text_sent_classifier.weight)
nn.init.xavier_uniform_(text_emot_classifier.weight)
nn.init.xavier_uniform_(image_classifier.weight)

## 3. Preprocessing & Prediction Functions

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_text_embedding(text):
    clean = clean_text(text)
    tokens = tokenizer(clean, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = bert_backbone(**tokens)
        embedding = torch.mean(outputs.last_hidden_state, dim=1)
    return embedding

def extract_acoustic_stats(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    
    feature_vector = np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        np.mean(mel, axis=1), np.std(mel, axis=1),
        np.mean(chroma, axis=1), np.std(chroma, axis=1),
        [np.mean(zcr), np.std(zcr)],
        [np.mean(centroid), np.std(centroid)]
    ])
    return feature_vector

image_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def preprocess_image(img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = image_transforms(img).unsqueeze(0).to(device)
    return tensor

def classify_review(text_input=None, audio_input_path=None, image_input_path=None):
    results = {}
    
    # Text inference
    if text_input:
        embedding = get_text_embedding(text_input)
        with torch.no_grad():
            feat_sent = text_sentiment_model(embedding)
            probs_sent = torch.softmax(text_sent_classifier(feat_sent), dim=-1).cpu().numpy()[0]
            
            feat_emot = text_emotion_model(embedding)
            probs_emot = torch.softmax(text_emot_classifier(feat_emot), dim=-1).cpu().numpy()[0]
            
        results['text'] = {
            'sentiment': sentiment_classes[np.argmax(probs_sent)],
            'sentiment_probs': probs_sent,
            'emotion': emotion_classes[np.argmax(probs_emot)],
            'emotion_probs': probs_emot
        }
        
    # Audio inference
    if audio_input_path and os.path.exists(audio_input_path):
        features = extract_acoustic_stats(audio_input_path).reshape(1, -1)
        features_scaled = audio_scaler.transform(features)
        probs_audio = audio_ensemble.predict_proba(features_scaled)[0]
        audio_emotions = ['Sadness', 'Neutral', 'Happiness']
        
        results['audio'] = {
            'sentiment': sentiment_classes[np.argmax(probs_audio)],
            'sentiment_probs': probs_audio,
            'emotion': audio_emotions[np.argmax(probs_audio)]
        }
        
    # Image inference
    if image_input_path and os.path.exists(image_input_path):
        image_tensor = preprocess_image(image_input_path)
        with torch.no_grad():
            feat_image = image_model(image_tensor)
            probs_image = torch.softmax(image_classifier(feat_image), dim=-1).cpu().numpy()[0]
        image_emotions = ['Sadness', 'Neutral', 'Happiness']
        
        results['image'] = {
            'sentiment': sentiment_classes[np.argmax(probs_image)],
            'sentiment_probs': probs_image,
            'emotion': image_emotions[np.argmax(probs_image)]
        }
        
    return results

## 4. Run Multimodal Inference Demo
We test the pipeline with input text and plot the prediction distributions.

In [ ]:
test_reviews = [
    "The product quality is amazing! I am so happy with this purchase 😊",
    "I am extremely disappointed and angry. The item arrived broken and customer service refused to help.",
    "The package arrived today. It is a brown cardboard box."
]

for review in test_reviews:
    pred = classify_review(text_input=review)
    print(f"\nReview: \"{review}\"")
    print(f"  --> Mapped Sentiment: {pred['text']['sentiment']}")
    print(f"  --> Mapped Emotion  : {pred['text']['emotion']}")

In [ ]:
# Plot probability bar charts
sample_review = "The product quality is amazing! I am so happy with this purchase 😊"
pred = classify_review(text_input=sample_review)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(x=sentiment_classes, y=pred['text']['sentiment_probs'], palette='viridis')
plt.title('Sentiment Prediction Probabilities')
plt.ylim(0, 1.0)
plt.ylabel('Confidence')

plt.subplot(1, 2, 2)
sns.barplot(x=emotion_classes, y=pred['text']['emotion_probs'], palette='magma')
plt.title('Emotion Prediction Probabilities')
plt.ylim(0, 1.0)
plt.ylabel('Confidence')

plt.tight_layout()
plt.show()